In [1]:
import os
import csv
import math
import math as _math
import cv2
import numpy as np
from tqdm import tqdm

In [2]:
PATH_INPUT = "Assets/"
KATEGORI = [
    "Normal",
    "kidneyStone"
]
OUTPUT_CSV  = "hasil_ekstraksi_percobaan1.csv"
GLCM_DISTANCE = 1
GLCM_ANGLES = [0,45,90,135]

In [3]:
def manual_clip(arr, a_min, a_max):
    if isinstance(arr, np.ndarray):
        flat = arr.flat
        return np.array([max(a_min, min(a_max, x)) for x in flat]).reshape(arr.shape)
    else:
        return max(a_min, min(a_max, arr))

In [4]:
def manual_arange(start, stop=None, step=1):
    if stop is None:
        stop = start
        start = 0
    length = int(math.ceil((stop - start) / step))
    return np.array([start + i*step for i in range(length)], dtype=np.float64)

In [5]:
def resize_manual(img, ukuran_baru):
    tinggi_lama, lebar_lama = img.shape
    lebar_baru, tinggi_baru = ukuran_baru
    img_f = img.astype(np.float64)

    skala_x = lebar_lama / lebar_baru
    skala_y = tinggi_lama / tinggi_baru

    y_asal = manual_clip(
        (manual_arange(tinggi_baru) + 0.5) * skala_y - 0.5,
        0, tinggi_lama - 1
    )
    x_asal = manual_clip(
        (manual_arange(lebar_baru) + 0.5) * skala_x - 0.5,
        0, lebar_lama - 1
    )

    hasil = np.zeros((tinggi_baru, lebar_baru), dtype=np.float64)

    for i in range(tinggi_baru):
        y = y_asal[i]
        y0 = int(math.floor(y))
        y1 = min(y0 + 1, tinggi_lama - 1)
        wy = y - y0
        for j in range(lebar_baru):
            x = x_asal[j]
            x0 = int(math.floor(x))
            x1 = min(x0 + 1, lebar_lama - 1)
            wx = x - x0

            p00 = img_f[y0, x0]
            p01 = img_f[y0, x1]
            p10 = img_f[y1, x0]
            p11 = img_f[y1, x1]

            atas  = p00 * (1 - wx) + p01 * wx
            bawah = p10 * (1 - wx) + p11 * wx
            hasil[i, j] = atas * (1 - wy) + bawah * wy

    return manual_clip(hasil, 0, 255).astype(np.uint8)

In [6]:
def build_glcm_manual(img, distance, angle_deg, levels=256):
    h, w = img.shape
    glcm = np.zeros((levels, levels), dtype=np.float64)

    if angle_deg == 0:
        di, dj = 0, distance
    elif angle_deg == 45:
        di, dj = -distance, distance
    elif angle_deg == 90:
        di, dj = -distance, 0
    elif angle_deg == 135:
        di, dj = -distance, -distance

    for i in range(h):
        for j in range(w):
            ni = i + di
            nj = j + dj
            if 0 <= ni < h and 0 <= nj < w:
                glcm[int(img[i, j]), int(img[ni, nj])] += 1

    glcm = glcm + glcm.T
    total = glcm.sum()
    if total > 0:
        glcm /= total
    return glcm

In [7]:
def ekstrak_fitur_glcm(glcm, levels=256):
    contrast = homogeneity = dissimilarity = entropy = asm = correlation = 0.0

    # Mean untuk korelasi
    mu_i = mu_j = 0.0
    for i in range(levels):
        for j in range(levels):
            mu_i += i * glcm[i, j]
            mu_j += j * glcm[i, j]

    # Standar deviasi untuk korelasi
    sigma_i = sigma_j = 0.0
    for i in range(levels):
        for j in range(levels):
            sigma_i += (i - mu_i) ** 2 * glcm[i, j]
            sigma_j += (j - mu_j) ** 2 * glcm[i, j]
    sigma_i = sigma_i ** 0.5
    sigma_j = sigma_j ** 0.5

    for i in range(levels):
        for j in range(levels):
            p = glcm[i, j]
            if p == 0:
                continue
            diff = i - j
            contrast      += (diff ** 2) * p
            homogeneity   += p / (1.0 + diff ** 2)
            dissimilarity += abs(diff) * p
            entropy       -= p * _math.log(p, 2)
            asm           += p ** 2
            if sigma_i > 0 and sigma_j > 0:
                correlation += ((i - mu_i) * (j - mu_j) * p) / (sigma_i * sigma_j)

    energy = asm ** 0.5
    return contrast, homogeneity, dissimilarity, entropy, asm, energy, correlation

In [8]:
def proses_glcm_satu_gambar(img, distance=1, angles=[0, 45, 90, 135]):
    row = {}
    for angle in angles:
        glcm = build_glcm_manual(img, distance, angle)
        c, h, d, e, a, en, cor = ekstrak_fitur_glcm(glcm)
        row[f"Contrast{angle}"]      = c
        row[f"Homogeneity{angle}"]   = h
        row[f"Dissimilarity{angle}"] = d
        row[f"Entropy{angle}"]       = e
        row[f"ASM{angle}"]           = a
        row[f"Energy{angle}"]        = en
        row[f"Correlation{angle}"]   = cor
    return row

In [9]:
CSV_HEADER = ["Filename", "Label"]

for _angle in GLCM_ANGLES:
    for _fitur in [
        "Contrast",
        "Homogeneity",
        "Dissimilarity",
        "Entropy",
        "ASM",
        "Energy",
        "Correlation"
    ]:
        CSV_HEADER.append(f"{_fitur}{_angle}")

In [ ]:
print("\nMemulai ekstraksi fitur GLCM ...")
semua_baris = []

for label in KATEGORI:
    folder = os.path.join(PATH_INPUT, label)
    if not os.path.exists(folder):
        print(f"Folder tidak ditemukan: {folder}")
        continue
    for nama_file in tqdm(sorted(os.listdir(folder)), desc=label):
        if not nama_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        jalur = os.path.join(folder, nama_file)
        img = cv2.imread(jalur, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = resize_manual(img,(256, 256))
        fitur_row = proses_glcm_satu_gambar(img,GLCM_DISTANCE,GLCM_ANGLES)
        baris = {"Filename": nama_file, "Label": label}
        baris.update(fitur_row)
        semua_baris.append(baris)

print("Jumlah data:", len(semua_baris))



Memulai ekstraksi fitur GLCM ...


Normal:  30%|███       | 30/100 [00:19<00:45,  1.55it/s]

In [ ]:
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_HEADER)
    writer.writeheader()
    writer.writerows(semua_baris)

print(f"{len(semua_baris)} data tersimpan ke: {OUTPUT_CSV}")

print(f"\n Selesai! CSV tersimpan di: {OUTPUT_CSV}")